# Rozdział 1 — Sygnał biomedyczny i tor pomiarowy

Eksperymenty E1.1–E1.6 odtwarzają figury i kontrole rozdziału.
Jedno rzeczywiste wejście znajduje się w katalogu danych. Przykłady
zmian toru wykorzystują model i mają znane odniesienie.

Rewizja D01 (12 listopada 2026): położenia adnotacji EKG pochodzą z oryginalnych plików WFDB `.atr`, v1.0.0. Próbki sygnału pozostają zgodne z wycinkami CPSB. Ślad korekty: `chapter_notes/korekta_adnotacji_D01.md`.


In [1]:
import csv
import json
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "data/ch01_manifest.json").is_file())
sys.path.insert(0, str(root / "src"))
from bspsbook.measurement import check_axis, file_digest, load_ecg, unwrap_ordered_counter

figures, outputs = root / "figures/generated", root / "results"
figures.mkdir(parents=True, exist_ok=True)
outputs.mkdir(exist_ok=True)
plt.rcParams.update({
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": .20, "legend.frameon": False,
    "svg.hashsalt": "CPSB-ch01",
})
blue, orange, gray = "#095b89", "#a44b24", "#545e66"
summary = {}

def save(fig, stem):
    fig.savefig(figures / f"{stem}.png", dpi=175, metadata={"Software": "CPSB"})
    fig.savefig(figures / f"{stem}.svg", metadata={"Date": None})
    plt.close(fig)

def gaussian(t, center, width):
    return np.exp(-.5 * ((t - center) / width) ** 2)

## E1.1 — Dwa kanały, różne punkty odniesienia

In [2]:
fs_model = 500
t = np.arange(3000) / fs_model
qrs = np.array([.72, 1.54, 2.39, 3.22, 4.15, 5.02])
delay = .220
device_delay = .080
arrivals = qrs + delay
recorded = arrivals + device_delay
ecg_model = np.zeros_like(t)
ppg = np.zeros_like(t)
ppg_delayed = np.zeros_like(t)
for r, start, later in zip(qrs, arrivals, recorded):
    ecg_model += (
        .12 * gaussian(t, r - .15, .025)
        - .15 * gaussian(t, r - .027, .009)
        + gaussian(t, r, .011)
        - .20 * gaussian(t, r + .030, .012)
        + .25 * gaussian(t, r + .235, .060)
    )
    for onset, trace in ((start, ppg), (later, ppg_delayed)):
        u = np.maximum((t - onset) / .115, 0)
        trace += u**2 * np.exp(2 * (1 - u))

np.testing.assert_allclose(arrivals - qrs, delay, atol=1e-12)
np.testing.assert_allclose(recorded - qrs, delay + device_delay, atol=1e-12)
np.testing.assert_allclose(np.diff(qrs), np.diff(recorded), atol=1e-12)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 4.8), layout="constrained")
axes[0].plot(t, ecg_model, color=blue, lw=1)
axes[0].set(ylabel="Model EKG [mV]", title="Znaczniki elektryczne oraz początek fali obwodowej")
axes[1].plot(t, ppg, color=blue, lw=1, label="Początek po 220 ms")
axes[1].plot(t, ppg_delayed, color=orange, lw=1, ls="--", label="Dodatkowe 80 ms toru")
for r, start, later in zip(qrs, arrivals, recorded):
    axes[0].axvline(r, color=gray, lw=.7, ls=":")
    axes[1].scatter([start, later], [0, 0], s=15, color=[blue, orange], zorder=4)
axes[1].set(xlabel="Czas [s]", ylabel="Model PPG [j.u.]", xlim=(.2, 5.85), ylim=(-.04, 1.20))
axes[1].legend(loc="upper right", ncol=2, fontsize=9)
save(fig, "ch01_01_ekg_ppg")
summary["E1.1"] = {"true_delay_ms": 220, "extra_device_delay_ms": 80,
                    "observed_delay_ms": 300, "successive_intervals_unchanged": True}
print(summary["E1.1"])

{'true_delay_ms': 220, 'extra_device_delay_ms': 80, 'observed_delay_ms': 300, 'successive_intervals_unchanged': True}


## E1.2 — Metadane i zmienność cykli w rzeczywistym EKG

In [3]:
record, manifest = load_ecg(root / "data")
fs = manifest["fs_hz"]
time, x = record["time_s"], record["signal_mV"].astype(float)
ref = record["r_samples"]
use = (time >= 12) & (time < 18)
assert record["valid_mask"][use].all()
segment = x[use]
seg_time = np.arange(len(segment)) / fs
selected = ref[use[ref]]
offsets = np.arange(-round(.18 * fs), round(.42 * fs) + 1)
beats = x[selected[:, None] + offsets]
rr = np.diff(selected) / fs
assert (len(segment), len(selected)) == (2160, 7)
assert np.ptp(rr) > 0

fig, axes = plt.subplots(2, 1, figsize=(8.3, 5.4), layout="constrained")
axes[0].plot(seg_time, segment, color=blue, lw=.95)
for n in selected:
    axes[0].axvline((n - 12 * fs) / fs, color=orange, lw=.7, alpha=.75)
axes[0].set(xlabel="Czas od początku wycinka [s]", ylabel="EKG [mV]", xlim=(0, 6))
for beat in beats:
    axes[1].plot(offsets / fs * 1000, beat, color=blue, lw=.9, alpha=.50)
axes[1].axvline(0, color=orange, lw=.9, ls="--")
axes[1].set(xlabel="Czas względem adnotacji [ms]", ylabel="EKG [mV]")
save(fig, "ch01_02_cykle")
summary["E1.2"] = {
    "samples": len(segment), "annotations": len(selected), "record_interval_s": [642, 648],
    "min_rr_s": float(rr.min()), "max_rr_s": float(rr.max()),
    "min_mV": float(segment.min()), "max_mV": float(segment.max()),
}
print(summary["E1.2"])

{'samples': 2160, 'annotations': 7, 'record_interval_s': [642, 648], 'min_rr_s': 0.7472222222222222, 'max_rr_s': 0.7888888888888889, 'min_mV': -0.6150000095367432, 'max_mV': 0.9200000166893005}


## E1.3 — Dwie zmiany obserwacji przy tym samym wejściu

In [4]:
offset = .36 * gaussian(seg_time, 2.45, .25)
line = .12 * np.sin(2 * np.pi * 50 * seg_time)
gain = 1 - .65 * gaussian(seg_time, 3.72, .28)
additive = segment + offset + line
multiplicative = gain * segment

np.testing.assert_allclose(additive - (offset + line), segment, atol=1e-12)
np.testing.assert_allclose(multiplicative / gain, segment, atol=1e-12)
physical_amplitude_change = gain * segment
unit_gain_measurement = physical_amplitude_change.copy()
np.testing.assert_array_equal(multiplicative, unit_gain_measurement)

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(8.3, 6.25), layout="constrained")
for ax, samples, color, title in zip(
    axes, (segment, additive, multiplicative), (blue, orange, orange),
    ("Fragment odniesienia", "Model addytywny · zmiana poziomu i 50 Hz",
     "Model multiplikatywny · lokalne osłabienie"),
):
    ax.plot(seg_time, samples, lw=.9, color=color)
    ax.set(ylabel="EKG [mV]", title=title, ylim=(-.78, 1.14))
axes[-1].set(xlabel="Czas od początku wycinka [s]", xlim=(0, 6))
save(fig, "ch01_03_zaklocenia")

def rms(v):
    return float(np.sqrt(np.mean(np.square(v))))

snr = 20 * np.log10(rms(segment) / rms(offset + line))
summary["E1.3"] = {"rms_reference_mV": rms(segment), "rms_added_mV": rms(offset + line),
                    "snr_db": float(snr), "gain_min": float(gain.min())}
print(summary["E1.3"])

{'rms_reference_mV': 0.34190264536682397, 'rms_added_mV': 0.12950386401528635, 'snr_db': 8.432394686927411, 'gain_min': 0.3500012794457048}


## E1.4 — Ograniczenie zakresu i utrata amplitudy

In [5]:
A, b_mV, limit_V = 500.0, 1.25, 1.0
u_linear = A * segment / 1000
u_offset = A * (segment + b_mV) / 1000
u_clipped = np.clip(u_offset, -limit_V, limit_V)
clipped = u_offset != u_clipped
recovered = 1000 * u_clipped / A - b_mV
assert np.max(np.abs(u_linear)) < limit_V and np.any(clipped)
np.testing.assert_allclose(recovered[~clipped], segment[~clipped], atol=1e-12)
assert np.all(recovered[clipped] < segment[clipped])
assert np.clip(A * (np.array([.80, .90]) + b_mV) / 1000, -limit_V, limit_V).tolist() == [1, 1]

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 5.0), layout="constrained")
axes[0].plot(seg_time, u_offset, color=gray, lw=.85, ls="--", label="Model liniowy")
axes[0].plot(seg_time, u_clipped, color=orange, lw=1, label="Wyjście z ograniczeniem")
axes[0].axhline(limit_V, color=gray, lw=.8, ls=":")
axes[0].set(ylabel="Napięcie toru [V]")
axes[0].legend(loc="lower right", ncol=2)
axes[1].plot(seg_time, segment, color=blue, lw=1, label="Fragment odniesienia")
axes[1].plot(seg_time, recovered, color=orange, lw=1, ls="--", label="Po odjęciu offsetu i przeliczeniu")
axes[1].scatter(seg_time[clipped], recovered[clipped], color=orange, s=10, zorder=4)
peak_time = seg_time[int(np.argmax(segment))]
axes[1].set(xlabel="Czas od początku wycinka [s]", ylabel="EKG [mV]",
            xlim=(peak_time - .15, peak_time + .23))
axes[1].legend(loc="upper right", ncol=2, fontsize=8.5)
axes[0].set_ylim(.25, 1.25)
axes[1].set_ylim(-.65, 1.17)
save(fig, "ch01_04_nasycenie")
summary["E1.4"] = {"clipped_samples": int(clipped.sum()),
                    "input_reference_peak_mV": float(segment.max()),
                    "reported_peak_mV": float(recovered.max())}
print(summary["E1.4"])

{'clipped_samples': 16, 'input_reference_peak_mV': 0.9200000166893005, 'reported_peak_mV': 0.75}


## E1.5 — Skala kodu i próbki utracone po akwizycji

In [6]:
def scale_uV_per_count(gain_value):
    return 4.5e6 / (gain_value * (2**23 - 1))

code_count = 5000
scale24, scale12 = scale_uV_per_count(24), scale_uV_per_count(12)
assert np.isclose(code_count * scale12, 2 * code_count * scale24)
packet_fs = 250
n = np.arange(75)
valid = (n < 31) | (n >= 38)
counters = ((242 + n) % 256)[valid]
restored_n = unwrap_ordered_counter(counters)
np.testing.assert_array_equal(restored_n, n[valid])
naive = np.arange(len(counters)) / packet_fs
correct = restored_n / packet_fs
missing = np.setdiff1d(n, restored_n)
assert len(missing) == 7
assert np.isclose(correct[-1] - naive[-1], .028)
counts = np.rint(code_count * np.sin(2 * np.pi * 6 * n / packet_fs)).astype(int)
signal_uV = counts * scale24
on_grid = np.full(n.shape, np.nan)
on_grid[restored_n] = signal_uV[valid]

fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.3, 4.9), layout="constrained")
axes[0].plot(1000 * naive, signal_uV[valid], "o-", color=orange, lw=.9, ms=2.8)
axes[0].set(title="Oś z numeru wiersza · brak znika z zapisu czasu", ylabel="Napięcie [µV]")
axes[1].plot(1000 * n / packet_fs, on_grid, "o-", color=blue, lw=.9, ms=2.8)
axes[1].axvspan(1000 * missing[0] / packet_fs, 1000 * restored_n[31] / packet_fs,
                color="#dfac82", alpha=.35, label="Siedem próbek bez danych")
axes[1].set(title="Oś z licznika · luka pozostaje widoczna", ylabel="Napięcie [µV]",
            xlabel="Czas od pierwszego pakietu [ms]", xlim=(0, 300))
axes[1].legend(loc="upper right", fontsize=9)
save(fig, "ch01_05_licznik")
with (outputs / "ch01_packet_example.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["row", "counter", "time_from_row_s", "time_from_counter_s", "signed_code", "voltage_uV"])
    writer.writerows(zip(range(len(counters)), counters, naive, correct, counts[valid], signal_uV[valid]))
summary["E1.5"] = {"code": code_count, "gain_24_uV": code_count * scale24,
                   "gain_12_uV": code_count * scale12, "missing_samples": len(missing),
                   "error_at_end_ms": round(float(1000 * (correct[-1] - naive[-1])), 6)}
print(summary["E1.5"])

{'code': 5000, 'gain_24_uV': 111.75872227653531, 'gain_12_uV': 223.51744455307062, 'missing_samples': 7, 'error_at_end_ms': 28.0}


## E1.6 — Kontrole i raport danych

In [7]:
def reject(call, *args):
    try:
        call(*args)
    except ValueError:
        return
    raise AssertionError("Kontrola nie odrzuciła niezgodnego wejścia.")

reject(check_axis, time + 1 / fs, fs, len(time))
reject(check_axis, time[1:], fs, len(time))
reject(unwrap_ordered_counter, [8, 8, 9])
np.testing.assert_array_equal(unwrap_ordered_counter([254, 255, 0, 1]), np.arange(4))
assert manifest["unit"] == "mV" and manifest["channel"] == "MLII"
summary["E1.6"] = {
    "axis_origin_checked": True, "length_checked": True, "counter_wrap_checked": True,
    "metadata": manifest,
    "invalid_samples_in_full_fragment": int((~record["valid_mask"]).sum()),
}
report = {
    "environment": {"python": platform.python_version(), "numpy": np.__version__,
                    "matplotlib": matplotlib.__version__},
    "data_manifest_sha256": file_digest(root / "data/ch01_manifest.json"),
    "helper_sha256": file_digest(root / "src/bspsbook/measurement.py"),
    "summary": summary,
    "outputs_sha256": {
        str(p.relative_to(root)): file_digest(p) for p in
        (*sorted(figures.glob("ch01_*")), outputs / "ch01_packet_example.csv")
    },
}
(outputs / "ch01_run.json").write_text(
    json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8",
)
print("Sprawdzono osie, jednostki, metadane, spójność adnotacji i zawijanie licznika.")

Sprawdzono osie, jednostki, metadane, spójność adnotacji i zawijanie licznika.
